# RetailMind AI - Day 5: Neural Networks & ANN (Keras)

Build a small feed-forward ANN for Telco churn, use early stopping, benchmark it against the saved classical champion, and save the ANN artifact.

In [1]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score


I0000 00:00:1794156491.413133 1306088 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794156491.456321 1306088 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794156493.072006 1306088 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 1. Reproducibility

In [2]:
tf.keras.utils.set_random_seed(42)


## 2. Load and clean the same Telco churn data

In [3]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1}).astype('float32')
print('Dataset:', X.shape)


Dataset: (7032, 19)


## 3. Fixed train/test split and ANN validation split

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)


Train: (4500, 19) Validation: (1125, 19) Test: (1407, 19)


## 4. Encode categoricals and normalise numerical inputs

The ANN receives the same type of tabular preprocessing used in earlier increments: scaled numerical features and one-hot encoded categorical features.

In [5]:
numeric_cols = X.select_dtypes(include='number').columns
categorical_cols = X.select_dtypes(exclude='number').columns
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])
X_train_nn = preprocessor.fit_transform(X_train).astype('float32')
X_val_nn = preprocessor.transform(X_val).astype('float32')
X_test_nn = preprocessor.transform(X_test).astype('float32')
print('ANN input features:', X_train_nn.shape[1])


ANN input features: 45


## 5. Build a small feed-forward ANN

In [6]:
ann = keras.Sequential([
    keras.layers.Input(shape=(X_train_nn.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
ann.summary()


E0000 00:00:1794156493.954794 1306088 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,057 (19.75 KB)

 Trainable params: 5,057 (19.75 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train with EarlyStopping

Validation loss is monitored to reduce overfitting on the tabular data. The best weights are restored automatically.

In [7]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
history = ann.fit(
    X_train_nn, y_train,
    validation_data=(X_val_nn, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)
print('Epochs run:', len(history.history['loss']))
print('Best validation loss:', round(min(history.history['val_loss']), 4))


Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7769 - loss: 0.4564 - val_accuracy: 0.7947 - val_loss: 0.4318
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8056 - loss: 0.4146 - val_accuracy: 0.7982 - val_loss: 0.4282
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8093 - loss: 0.4095 - val_accuracy: 0.7991 - val_loss: 0.4277
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8091 - loss: 0.4057 - val_accuracy: 0.7991 - val_loss: 0.4288
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8096 - loss: 0.4025 - val_accuracy: 0.8000 - val_loss: 0.4293
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8104 - loss: 0.3995 - val_accuracy: 0.7982 - val_loss: 0.4295
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8118 - loss: 0.3967 - val_accuracy: 0.8000 - val_loss: 0.4310
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8116 - loss: 0.3943 - val_accu

## 7. Evaluate ANN on the untouched test set

In [8]:
ann_prob = ann.predict(X_test_nn, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype(int)
ann_metrics = {
    'Accuracy': accuracy_score(y_test, ann_pred),
    'Precision': precision_score(y_test, ann_pred),
    'Recall': recall_score(y_test, ann_pred),
    'ROC-AUC': roc_auc_score(y_test, ann_prob)
}
pd.DataFrame([ann_metrics], index=['ANN']).round(4)


,Accuracy,Precision,Recall,ROC-AUC
ANN,0.7946,0.6557,0.4786,0.8337


## 8. Benchmark against the saved classical champion

Day 5 compares the ANN against the versioned classical champion from Day 3.

In [9]:
classical = joblib.load('../models/churn_v3.joblib')
classical_pred = classical.predict(X_test)
classical_prob = classical.predict_proba(X_test)[:, 1]
classical_metrics = {
    'Accuracy': accuracy_score(y_test, classical_pred),
    'Precision': precision_score(y_test, classical_pred),
    'Recall': recall_score(y_test, classical_pred),
    'ROC-AUC': roc_auc_score(y_test, classical_prob)
}
comparison = pd.DataFrame([classical_metrics, ann_metrics], index=['Classical Champion', 'Keras ANN'])
comparison.round(4)


,Accuracy,Precision,Recall,ROC-AUC
Classical Champion,0.7896,0.6021,0.6150,0.8289
Keras ANN,0.7946,0.6557,0.4786,0.8337


## 9. Save the ANN and its preprocessing artifact

In [10]:
ann.save('../models/churn_ann.keras')
joblib.dump(preprocessor, '../models/churn_ann_preprocessor.joblib')
print('Saved: ../models/churn_ann.keras')
print('Saved: ../models/churn_ann_preprocessor.joblib')


Saved: ../models/churn_ann.keras
Saved: ../models/churn_ann_preprocessor.joblib


In [11]:
print('DAY 5 - ANN BENCHMARK')
print('-' * 60)
print(comparison.round(4).to_string())
print('\nEpochs run:', len(history.history['loss']))
print('Best validation loss:', f"{min(history.history['val_loss']):.4f}")
print('ANN artifact: models/churn_ann.keras')
print('ANN preprocessor: models/churn_ann_preprocessor.joblib')


DAY 5 - ANN BENCHMARK
------------------------------------------------------------
                    Accuracy  Precision  Recall  ROC-AUC
Classical Champion    0.7896     0.6021  0.6150   0.8289
Keras ANN             0.7946     0.6557  0.4786   0.8337

Epochs run: 8
Best validation loss: 0.4277
ANN artifact: models/churn_ann.keras
ANN preprocessor: models/churn_ann_preprocessor.joblib
